In [ ]:
from suite2p import default_ops
from suite2p.registration import register
import tifffile as tiff
import matplotlib.pyplot as plt
import numpy as np
import time
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift, white_tophat
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import closing, square
from skimage.segmentation import relabel_sequential
import matplotlib.animation as animation
from IPython.display import Image, display
from matplotlib.patches import Rectangle

In [ ]:
data_path1 = 'data/Nick_zoom8_512x512_20pct_8planes_1um_RO7_150um_NlightG2_tdTomato_00001_00002__Substack_3999-7999-2.tif'

In [ ]:
ops = default_ops()

In [ ]:
ops['batch_size'] = 250
ops['nimg_innit'] = 250

In [ ]:
frames1 = tiff.imread(data_path1).astype(np.float32)
frames1.shape, frames1.dtype

In [ ]:
# Frames1 is a time series of z-stacks, each stack consisting of 8 frames. Need to average each volume of 8 frames together.

N_frames, height, width = frames1.shape
num_vols = N_frames // 8  

# Cut excess frames
frames1 = frames1[: num_vols * 8] 

# Reshape so each vol of 8 frames is in its own sub-array (shape = num_vols, 8, height, width)
frames1_reshaped = frames1.reshape(num_vols, 8, height, width)

# Take the top n planes in each volume
n_planes=8
frames1_reshaped = frames1_reshaped[:, :n_planes, :, :]

# Average over axis=1 (shape = num_vols, height, width)
frames1_avg = frames1_reshaped.mean(axis=1)

frames1_avg = frames1_avg - frames1_avg.min()
frames1_avg.shape

# Rigid registration 

In [ ]:
def align_frames(frames, n_sample_frames=250, n_top_frames=20):
    """
    Apply rigid motion correction to a stack of frames by:
    - Taking a random subset of frames
    - Taking frame that is most correlated to all others as seed frame
    - Averaging the seed frame with the 19 other frames it's most correlated with to produce reference frame
    - Align each frame with the reference frame using phase cross correlation
    
    Parameters:
        frames (np.ndarray): 3D array of frames with shape (N, H, W).
    
    Returns:
        reference_frame (np.ndarray): The computed reference frame - a 2D array with shape (H, W).
        aligned_frames (np.ndarray): 3D array of frames aligned to the reference frame with shape (N, H, W).
    """

    ## Calculate ref frame
    t0 = time.time()
    
    # Take up to 250 random frames
    if frames.shape[0] < n_sample_frames:
        size = frames.shape[0]
    else:
        size = n_sample_frames
    random_indices = np.random.choice(len(frames), size=size, replace=False)
    sample_frames = frames[random_indices]

    # Flatten each frame into 1D
    sample_frames_flat = sample_frames.reshape(size, -1)
    
    # Compute correlation matrix of each frame with all other frames
    corr_matrix = np.corrcoef(sample_frames_flat)
    
    # Avg the correlation of each frame with all other frames
    avg_corr = np.mean(corr_matrix, axis=1)
    
    # Get frame with highest average correlation
    seed_idx = np.argmax(avg_corr)
    seed_frame = sample_frames[seed_idx]
    
    # Average top n frames most correlated with seed frame
    seed_corr = corr_matrix[seed_idx].copy() # Get correlations of seed frame with all other frames
    sorted_indices = np.argsort(seed_corr)[::-1] # Sort indices, starting from the index of the frame with the highest correlation to seed
    top_indices = sorted_indices[:n_top_frames] # Take top n indices (default = 20)
    top_frames = sample_frames[top_indices] # Get top n frames
    reference_frame = np.median(top_frames, axis=0) # Average top n most correlated frames to produce reference frame
    
    t1 = time.time()
    print(f'Computed reference frame in {t1-t0:.2f} s\n')

    ## Align frames to  ref frame
    t0 = time.time()
    
    # Create new array of frames, each aligned to the reference image by phase cross correlation
    aligned_frames = np.empty_like(frames) 
    for i in range(frames.shape[0]):
        curr_frame = frames[i]
        
        # Use phase correlation to identify the relative shift between the reference frame and current frame
        shift_vector, error, diffphase = phase_cross_correlation(reference_frame, curr_frame)
        
        # Apply the shift to the current frame
        aligned_frame = shift(curr_frame, shift=shift_vector).copy()
        aligned_frames[i] = aligned_frame
        
    t1 = time.time()
    print(f'Aligned {frames.shape[0]} frames in {t1-t0:.2f} s\n')

    return reference_frame, aligned_frames

In [ ]:
def generate_crop_indices(frames):
    """
    Crops frames by removing outer cols and rows that are 0 in all frames. This removes edge artifacts that are a result of rigid registration.

    Parameters:
        frames (np.ndarray): 3D array of frames with shape (N, H, W).

    Returns:
        crop_indices (tuple of 4 ints): top, bottom, left, right indices used for cropping.
    """
    t0 = time.time()
    height = frames.shape[1]
    width = frames.shape[2]
    
    # Create mask that is True where all frames are > 0 at each pixe;
    mask = np.all(frames > 0, axis=0)  
    
    # Get coords of all pixels that are non-zero in every frame
    rows, cols = np.where(mask)

    # If no pixel is nonzero in every frame, don't crop (as below part of func won't work as intended)
    if len(rows) == 0 or len(cols) == 0:
        return (0, -1, 0, -1)
    
    # Otherwise, calculate the the crop indices (top, bottom, left, right)
    # These are the first rows/cols where a pixel is non-zero in every frame.
    # This will crop out edges that have pixels with zero values in one or more frames likely due to edge artifacts.
    top, bottom = rows.min(), rows.max()
    left, right = cols.min(), cols.max()

    top_crop = top
    bottom_crop = height - bottom
    left_crop = left
    right_crop = width - right

    t1 = time.time()

    print(f'Computed crop in {t1-t0:.2f} s')

    print(f'Crop box: ({top}, {bottom}, {left}, {right})')
    print(f'Cropped: {top_crop} rows from top, {bottom_crop} rows from bottom, {left_crop} cols from left, {right_crop} cols from right')
    print(f'Old Dimensions: {height} x {width}')
    print(f'New Dimensions: {height - top_crop - bottom_crop + 1} x {width - left_crop - right_crop + 1}\n')

    return (top, bottom, left, right)

In [ ]:
def crop(frames, crop_indices):
    """
    Crops each frame in frames using the indices generated by generate_crop_indices.

    Parameters:
        frames (np.ndarray): 3D array of frames with shape (N, H, W).
        crop_indices (tuple of 4 ints): top, bottom, left, right indices used for cropping.
        
    Returns:
        cropped_frames (np.ndarray): The cropped 3D array of shape (N, bottom-top, right-left).
    """
    top, bottom, left, right = crop_indices
    if frames.ndim == 3:
        cropped_frames = frames[:, top:bottom+1, left:right+1].copy()
    else:
        cropped_frames = frames[top:bottom+1, left:right+1].copy()
    return cropped_frames

In [ ]:
def iterative_align_crop(unaligned_frames, n_iterations):
    """
    Iteratively compute reference frame, align frames to reference, crops edge artifacts, then repeats for n_iterations.
    NB: there is minimal improvement in alignment accuracy between one vs five iterations, but substantial time increase.
    2 iterations seems optimal trade off between time/accuracy.
    
    Parameters:
        n_iterations (int): Number of iterations to run.
        unaligned_frames (np.ndarray): 3D array with shape (N, H, W).
    
    Returns:
        crop_ref (np.ndarray): A 2D array of the cropped computed reference frame.
        crop_original (np.ndarray): A crop of the original 3D array of frames.
        crop_aligned (np.ndarray): A crop of the 3D array of frames aligned to the reference frame.
    """
    iter_t0 = time.time()

    crop_indices = {}
    for i in range(n_iterations):
        print(f'Starting iteration {i + 1}...\n')
        if i == 0:
            ref_frame, aligned_frames = align_frames(unaligned_frames)
            indices = generate_crop_indices(aligned_frames)
            crop_ref = crop(ref_frame, indices)
            crop_original = crop(unaligned_frames, indices)
            crop_aligned = crop(aligned_frames, indices)
        else:
            ref_frame, aligned_frames = align_frames(crop_aligned)
            indices = generate_crop_indices(aligned_frames)
            crop_ref = crop(crop_ref, indices)
            crop_original = crop(crop_original, indices)
            crop_aligned = crop(aligned_frames, indices)
            
        crop_indices[i] = indices
    
    iter_t1 = time.time()
    print(f'Total time for {n_iterations} iterations {iter_t1 - iter_t0:.2f} s\n')
            
    return crop_ref, crop_original, crop_aligned, crop_indices

# Suite2p registration

In [ ]:
# This code func runs registration using the suite2p library - https://github.com/MouseLand/suite2p/blob/main/suite2p/registration/register.py
def align_frames_suite2p(frames, ops):
    """
    Uses Suite2Ps compute_reference_and_register_frames() func to do alignment.

    Parameters:
    frames (np.ndarray): 3D array of frames with shape (N, H, W).
    ops (dict): dictionary describing the settings to use use for the suite2p pipeline
    
    Returns:
        reference_frame (np.ndarray): The computed reference frame - a 2D array with shape (H, W).
        aligned_frames (np.ndarray): 3D array of frames aligned to the reference frame with shape (N, H, W).
    """
    t0 = time.time()
    
    frames_copy = frames.copy()
    reference_frame, rmin, rmax, mean_img, rigid_offsets, nonrigid_offsets, zpos_cmax = register.compute_reference_and_register_frames(frames_copy, ops=ops)
    yoff = rigid_offsets[0]
    xoff = rigid_offsets[1]
    
    aligned_frames = np.empty_like(frames)
    for i, frame in enumerate(frames):
        aligned_frames[i] = np.roll(frame, (-yoff[i], -xoff[i]), axis=(0, 1))

    t1 = time.time()
    print(f'Total runtime {t1-t0:.2f} s')
    
    return reference_frame, aligned_frames

# Check alignment

In [ ]:
def compare_corr(unaligned_frames, aligned_frames):
    """
    Computes a mean correlation score for the unaligned and aligned frame series.
    
    Parameters:
        unaligned_frames (np.ndarray): 3D array with shape (N, H, W).
        aligned_frames (np.ndarray): 3D array with shape (N, H, W).
    
    Returns:
        result_string: String containing the mean correlations of the unaligned and aligned series.

    """

    # Compute correlation matrix of each frame with all other frames
    size = unaligned_frames.shape[0]
    mean_corr_unaligned = np.mean(np.corrcoef(unaligned_frames.reshape(size, -1)), axis=1).mean()
    mean_corr_aligned = np.mean(np.corrcoef(aligned_frames.reshape(size, -1)), axis=1).mean()

    result_string = f'Mean Unaligned Corr: {mean_corr_unaligned:.3f}\nMean Aligned Corr:   {mean_corr_aligned:.3f}\nDifference:          {mean_corr_aligned - mean_corr_unaligned:.3f}'
    
    return result_string

In [ ]:
def visualise_alignment(ref_frame, unaligned_frame, aligned_frame):
    """
    Visualise motion correction by comparing an unaligned and aligned frame to the reference image and computing the diff.
    
    Parameters:
        ref_frame (np.ndarray): 2D array with shape (H, W).
        unaligned_frame (np.ndarray): 2D array with shape (H, W).
        aligned_frame (np.ndarray): 2D array with shape (H, W).
    
    Returns:
        Nothing, but makes a 3x2 plot visualising the alignment.

    """
    
    # Comp difference matrix
    unaligned_diff = unaligned_frame - ref_frame
    aligned_diff = aligned_frame - ref_frame 

    # Comp correlations
    unaligned_corr = np.corrcoef(ref_frame.ravel(), unaligned_frame.ravel())[0, 1]
    aligned_corr = np.corrcoef(ref_frame.ravel(), aligned_frame.ravel())[0, 1]

    # Get max absolute diff to scale color map
    max_diff = max(np.abs(unaligned_diff).max(), np.abs(aligned_diff).max())
    vmin, vmax = -max_diff, max_diff

    plt.figure(figsize=(14, 14))

    img_cmap='viridis'
    diff_cmap='bwr'
    
    # Reference frame
    plt.subplot(3, 2, 1)
    plt.imshow(ref_frame, cmap=img_cmap)
    plt.title("Reference")
    plt.axis("off")

    # Reference frame
    plt.subplot(3, 2, 2)
    plt.imshow(ref_frame, cmap=img_cmap)
    plt.title("Reference")
    plt.axis("off")
    
    # Unaligned frame
    plt.subplot(3, 2, 3)
    plt.imshow(unaligned_frame, cmap=img_cmap)
    plt.title("Unaligned Frame")
    plt.axis("off")
    
    # Aligned frame
    plt.subplot(3, 2, 4)
    plt.imshow(aligned_frame, cmap=img_cmap)
    plt.title("Aligned Frame")
    plt.axis("off")
    
    # Unaligned diff
    plt.subplot(3, 2, 5)
    plt.imshow(unaligned_diff, cmap=diff_cmap, vmin=vmin, vmax=vmax)
    plt.title(f"Unaligned Difference (Corr: {unaligned_corr:.3f})")
    plt.colorbar(label="Intensity difference")
    plt.axis("off")
    
    # Aligned diff
    plt.subplot(3, 2, 6)
    plt.imshow(aligned_diff, cmap=diff_cmap, vmin=vmin, vmax=vmax)
    plt.title(f"Aligned Difference (Corr: {aligned_corr:.3f})")
    plt.colorbar(label="Intensity difference")
    plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
def show(frame):
    """
    Render a 2D matrix as an image.

    Parameters:
        frame (np.ndarray): 2D array with shape (H, W).

    Returns:
        Nothing, just plots an image.
    """
    plt.figure(figsize=(8,8))
    plt.imshow(frame, vmax=frame.max(), vmin=frame.min())
    plt.colorbar()
    plt.axis('off')
    plt.show();

In [ ]:
def roll_avg(aligned_frames, roll_size=3):
    """
    Averages intensity data in each pixel using a rolling average.

    Parameters:
        aligned_frames (np.ndarray): 3D array with shape (N, H, W).
        roll_size (int): Number of frames to average over.

    Returns:
        roll_avg_frames (np.ndarray): 3D array with shape (N - roll_size + 1, H, W) where each frame is the rolling average of roll_size frames.
    """
    N = aligned_frames.shape[0]

    roll_avg_frames = np.array([
        np.mean(aligned_frames[i:i + roll_size], axis=0)
        for i in range(N - roll_size + 1)
    ])
    return roll_avg_frames

In [ ]:
def make_gif(frames, file_name='test', interval=50, show=True):
    """
    Saves and optionally renders a gif of the frames array.

    Parameters:
        frame (np.ndarray): 3D array with shape (N, H, W).
        file_name (str): file name for saved gif
        interval (int): msec interval between frames
        show (bool): whether or not to render the gif in the notebook

    Returns:
        Nothing.
    """

    file_name = file_name + '.gif'
    
    fig, ax = plt.subplots()
    im = ax.imshow(frames[0], cmap='viridis', vmin=frames.min(), vmax=frames.max())
    cbar = fig.colorbar(im, ax=ax)
    ax.set_title('Frame 0')
    ax.axis('off')
    
    def update(frame_idx):
        im.set_data(frames[frame_idx])
        ax.set_title(f'Frame {frame_idx}')
        return [im]
    
    anim = animation.FuncAnimation(fig, update, frames=frames.shape[0], interval=interval, blit=True)
    anim.save(file_name, writer='imagemagick')
    plt.close(fig)
    
    if show:
        display(Image(filename=file_name))

In [ ]:
def make_compar_gif(frames1, frames2, name1='Frames1', name2='Frames2', file_name='test', interval=50, show=True):
    """
    Saves and optionally renders a gif of two frames arrays side by side.

    Parameters:
        frames1 (np.ndarray): 3D array with shape (N, H, W).
        frames2 (np.ndarray): 3D array with shape (N, H, W).
        name1 (str): title for first frames array
        name2 (str): title for second frames array
        file_name (str): file name for saved gif
        interval (int): msec interval between frames
        show (bool): whether or not to render the gif in the notebook

    Returns:
        Nothing.
    """

    file_name = file_name + '.gif'
    
    fig, axs = plt.subplots(1, 2, figsize=(8, 4),constrained_layout=True)
    
    im1 = axs[0].imshow(frames1[0], cmap='viridis', vmin=frames1.min(), vmax=frames1.max())
    # cbar = fig.colorbar(im1, ax=axs[0], location='left')
    axs[0].set_title(f'{name1}: Frame 0')
    axs[0].axis('off')

    im2 = axs[1].imshow(frames2[0], cmap='viridis', vmin=frames2.min(), vmax=frames2.max())
    # cbar = fig.colorbar(im2, ax=axs[1], location='right')
    axs[1].set_title(f'{name2}: Frame 0')
    axs[1].axis('off')
    
    def update(frame_idx):
        im1.set_data(frames1[frame_idx])
        axs[0].set_title(f'{name1}: Frame {frame_idx}')

        im2.set_data(frames2[frame_idx])
        axs[1].set_title(f'{name2}: Frame {frame_idx}')

        
        return [im1, im2]
    
    anim = animation.FuncAnimation(fig, update, frames=frames1.shape[0], interval=interval, blit=True)
    anim.save(file_name, writer='imagemagick')
    plt.close(fig)
    
    if show:
        display(Image(filename=file_name))

In [ ]:
# Suite2p alignment
ref_s2p, aligned_s2p = align_frames_suite2p(frames1_avg, ops)

In [ ]:
# Custom alignment
ref, frames_cropped, aligned, crop_indices = iterative_align_crop(frames1_avg, 2)

In [ ]:
# Apply same crop to the suite2p registered frame to allow comparison
ref_s2p_cropped = crop(ref_s2p, crop_indices[0]) 
ref_s2p_cropped = crop(ref_s2p_cropped, crop_indices[1]) 
aligned_s2p_cropped = crop(aligned_s2p, crop_indices[0]) 
aligned_s2p_cropped = crop(aligned_s2p_cropped, crop_indices[1]) 

In [ ]:
print(compare_corr(frames_cropped, aligned_s2p_cropped))

In [ ]:
print(compare_corr(frames_cropped, aligned))

In [ ]:
visualise_alignment(ref_s2p_cropped, frames_cropped[50], aligned_s2p_cropped[50])

In [ ]:
visualise_alignment(ref_s2p_cropped, frames_cropped[50], aligned[50])

In [ ]:
unaligned_avg = roll_avg(frames1_avg, roll_size=3)
aligned_avg = roll_avg(aligned, roll_size=3)
make_compar_gif(unaligned_avg, aligned_avg, name1='Unaligned', name2='Aligned', file_name='unaligned_aligned')

In [ ]:
# Save Files
tiff.imwrite("aligned_20_03.tif", aligned, imagej=True)
tiff.imwrite("original_20_03.tif", frames_cropped, imagej=True)

# Signal analysis

In [ ]:
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.animation as animation
from IPython.display import Image, display

In [ ]:
def make_tiles(frames, tile_size=20, blurr_size=2):
    """
    Break frames into tiles of size tile_size and after applying an optional gaussian blur with sigma blurr_size.
    Calculates the mean signal in each tile and returns that as the new tiled_frames.

    Parameters:
         frames (np.ndarray): 3D array with shape (N, H, W).
         tile_size (int): size in pixels of each tile
         blurr_size (float): adds optional gaussian blur to frames before before tiling.
        
    Returns:
        tiled_frames (np.ndarray): 3D array with shape (N, num_tiles_y, num_tiles_x).
        
    """
    nframes, height, width = frames.shape
    rows = height // tile_size
    cols = width // tile_size

    if blurr_size:
        frames = gaussian(aligned, sigma=(0, blurr_size, blurr_size))
        
        # If tile_size == 1, return here as no tiling need be applied
        if tile_size == 1:
            return frames
    
    tiled_frames = np.zeros((nframes, rows, cols))
    
    for frame in range(nframes):
        for row in range(rows):
            for col in range(cols):
                roi = aligned[frame, row * tile_size:(row + 1)*tile_size, col * tile_size:(col + 1)*tile_size] # Calculate mean int across ROI of original frame
                tiled_frames[frame, row, col] = np.mean(roi) # Save that mean int into new tiled frame

    return tiled_frames

In [ ]:
def overlay_rois(frames, tiles_to_draw, tile_size):
    '''
    Takes in an array of frames to generate an average background image, and overlays on that the True tiles given in tiles array, of size tile_size.

    Parameters:
        frames1 (np.ndarray): 3D array with shape (N, H, W).
        tiles_to_draw (np.ndarray): 2D array with shape (H//tile_size, W//tile_size). Array is filled with bools where True indicates tiles to draw.
        tile_size (int): size of each tile

    Returns:
        Noting, but plots the True tiles overlaid on the average frame.
    '''
    # height, width = tiles.shape
    # rows = height // tile_size
    # cols = width // tile_size

    rows, cols = tiles_to_draw.shape

    avg_img = np.mean(frames, axis=0)

    fig, ax = plt.subplots(figsize=(14,14))
    ax.imshow(avg_img, cmap='viridis')

    for row in range(rows):
        for col in range(cols):
            if tiles_to_draw[row, col]:
                rect = patches.Rectangle(
                    (col * tile_size, row * tile_size),
                    tile_size,
                    tile_size,
                    linewidth=2,
                    edgecolor='red',
                    facecolor='none'
                )
                ax.add_patch(rect)
                ax.axis('off')

    plt.show()

In [ ]:
def plot_rois(frames_ts, tiles_to_plot, tile_size=25, frame_rate=7.5):
    '''
    Plots a ∆F/F timeseries for each tile ROI given in tiles_to_plot using the data in frames_ts.
    
    Parameters:
        frames_ts (np.ndarray): 3D array with shape (N, H, W). The timeseries data is taken from this array
        tiles_to_plot (np.ndarray): 2D array with shape (H//tile_size, W//tile_size). Array is filled with bools where True indicates tiles to draw.
        tile_size (int): size of each tile
        frame_rate (int): frame_rate of frames_ts. Used to convert frames to times for x-axis.

    Returns
        Nothing, but plots the timeseries from the frames array at each ROI.
    '''
    # Convert frames to time: frame * 1/frame_rate = time of frame
    nframes = frames_ts.shape[0]
    times = np.arange(nframes) * (1/frame_rate)
    
    rows, cols = tiles_to_plot.shape
    for row in range(rows):
        for col in range(cols):
            if tiles_to_plot[row, col]:
                time_series = frames_ts[:, row, col]
                plt.figure(figsize=(8, 4))
                plt.plot(times, time_series*100, color='red')
                plt.title(f"Time series for tile at row {row}, col {col}")
                plt.xlabel("Time (seconds)")
                plt.ylabel("∆F/F (%)")
                plt.gca().spines[['top', 'right']].set_visible(False)
                plt.ylim([-50, 200])
                plt.xlim([0, 33])
                plt.tight_layout()
                plt.show()

In [ ]:
def overlay_plotted_rois(frames_og, frames_ts, tiles_to_plot, ymax=2, tile_size=25):
    '''
    Parameters:
        frames_og (np.ndarray): 3D array with shape (N, H, W). This is the original array used to calculate the average bg image.
        frames_ts (np.ndarray): 3D array with shape (N, H // tile_size, W // tile_size). The timeseries data is taken from this array.
        tiles_to_plot (np.ndarray): 2D array with shape (H // tile_size, W // tile_size). Array is filled with bools where True indicates tiles to draw.
        tile_size (int): size of each tile

    Returns:
        Nothing, but plots the timeseries of each ROI and overlays them on the average of frames as a background image.
    '''

    nframes, width, height = frames_og.shape
    rows, cols = tiles_to_plot.shape
    fig, ax = plt.subplots(figsize=(14, 14))

    # Show background image as average of frames_og
    ax.imshow(
        np.mean(frames_og, axis=0),
        cmap='viridis',
        extent=[0, width, height, 0],
        origin='upper',
        aspect='auto'
    )

    ax.set_title("∆F/F timeseries for significant ROIs")

    # Manual offsets needed as for some reason the ROIs aren't aligning with the background image correctly
    col_offset = - 3.15
    row_offset = - 3.35

    # Plot each tile in tiles_to_plot where value of tile = True
    for row in range(rows):
        for col in range(cols):
            if tiles_to_plot[row, col]:
                time_series = frames_ts[:, row, col]
                inset_ax = inset_axes(
                    ax,
                    width='100%',
                    height='100%',
                    bbox_to_anchor = (col * tile_size + col_offset, row * tile_size + row_offset, tile_size, tile_size),
                    bbox_transform = ax.transData,
                    loc='upper left'
                )
                
                inset_ax.plot(time_series, color='yellow')
                inset_ax.set_facecolor('None')
                inset_ax.patch.set_alpha(1)
                inset_ax.set_xticks([])
                inset_ax.set_yticks([])
                inset_ax.set_ylim([-0.5, ymax])
                inset_ax.set_xlim([0, nframes])

                # Draw square ROI 
                for spine in inset_ax.spines.values():
                    spine.set_edgecolor('red')
                    
    ax.axis('off')
    plt.show()

In [ ]:
def intensity_time_map(frames, tiles_to_plot=None, tile_size=25):
    '''
    Plots a map visualising the time at which each pixel exhibits peak intensity.
    
    Parameters:
        frames (np.ndarray): 3D array with shape (N, H, W).
        tiles_to_plot (np.ndarray): 2D array with shape (H // tile_size, W // tile_size). Array is filled with bools where True indicates tiles to draw.
        tile_size (int): size of each tile

    Returns:
        Nothing, but plots intensity timemap with specified ROI tiles optionally overlaid.
    '''
    nframes, width, height = frames.shape
    argmax_idx = np.argmax(frames, axis=0) # Frame index where each pixel is brightest
    norm_idx = argmax_idx / (frames.shape[0] - 1) # Normalise indices

    # Create cmap 
    a = '#ef476f'
    b = '#ffd166'
    c = '#9bc1bc'
    cmap = LinearSegmentedColormap.from_list(
        'custom_cmap',
        [(0, a), (0.5, b), (1, c)]
    )

    fig, ax = plt.subplots(figsize=(14,14))
    im = ax.imshow(
        norm_idx,
        cmap=cmap,
        vmin=0,
        vmax=1,
        extent=[0, width, height, 0],
        origin='upper',
        aspect='auto'
    )
    
    ax.set_title('Time of Peak Intensity')
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, fraction=0.06125, pad=0.025, aspect=15) # This is messy, but tried to manually align cbar
    cbar.set_label('0 = Early, 1 = Late')

    # Optionally overlay tile ROIs if passed as argument
    if np.any(tiles_to_plot):
        rows, cols = tiles_to_plot.shape
        for row in range(rows):
            for col in range(cols):
                if tiles_to_plot[row, col]:
                    rect = patches.Rectangle(
                        (col * tile_size, row * tile_size),
                        tile_size,
                        tile_size,
                        linewidth=2,
                        edgecolor='red',
                        facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.axis('off')

In [ ]:
def make_overlaid_timeseries_gif(frames, tiled, tiles_to_plot, tile_size=25, file_name='test', interval=50, show=True, ymax=2):
    """
    Saves a gif of frames with inset ROI time-series for significant tiles.
    NB: this function is slow to execute, especially if there are lots of tiles to plot!

    Parameters:
        frames (np.ndarray): 3D array with shape (N, H, W).
        tiled (np.ndarray):  3D array (N, H // tile_size, W // tile_size), containing timeseries data averaged by ROI.
        significant_tiles (np.ndarray): 2D array (H // tile_size, W // tile_size) of bools where True = tile to plot.
        tile_size (int): size of each tile in pixels
        file_name (str): filename for the output gif.
        interval (int): ms between frames in the animation.
        show (bool): whether to display the gif in notebook.

    Returns:
        Nothing, saves a .gif and optionally displays it.
    """


    nframes, height, width = frames.shape
    rows, cols = significant_tiles.shape

    fig, ax = plt.subplots(figsize=(14,14))
    im = ax.imshow(frames[0], cmap='viridis', vmin=frames.min(), vmax=frames.max())
    ax.set_title('Frame 0')
    ax.axis('off')

    lines = []
    roi_positions = []

    # Have to manually add offsets cos ROIs aren't aligning properly
    col_offset = -3.15
    row_offset = - 3.35

    # Create inset axes + line objects for each significant tile
    for row in range(rows):
        for col in range(cols):
            if tiles_to_plot[row, col]:
                # Create an inset ax at ROI tile (row, col)
                axins = inset_axes(
                    ax,
                    width="100%",  
                    height="100%", 
                    bbox_to_anchor=(col * tile_size + col_offset, row * tile_size + row_offset, tile_size, tile_size),
                    bbox_transform=ax.transData,
                    loc='upper left'
                )

                axins.set_facecolor("none")
                axins.patch.set_alpha(1)

                for spine in axins.spines.values():
                    spine.set_edgecolor('red')
                    
                axins.set_xticks([])
                axins.set_yticks([])
                
                # Create a line in this inset ax
                line, = axins.plot([], [], color='yellow', alpha=1.0)
                
                # Store references for updating
                lines.append(line)
                roi_positions.append((row, col))

    def update(frame_idx):
        """
        Update function for FuncAnimation.
        """
        # Update main image
        im.set_data(frames[frame_idx])
        ax.set_title(f'Frame {frame_idx}')
        
        # Update each ROI's timeseries plot
        for line, (row, col) in zip(lines, roi_positions):
            # The entire time series for that tile
            time_series = tiled[:, row, col]
            # Plot up to current frame
            line.set_data(np.arange(frame_idx+1), time_series[:frame_idx+1])
            
            axins = line.axes
            axins.set_xlim(0, nframes)
            axins.set_ylim(-0.5, ymax)
        
        return [im] + lines

    # Create animation
    anim = animation.FuncAnimation(
        fig,
        update,
        frames=nframes,
        interval=interval,
        blit=False  
    )

    # Save as gif 
    gif_file = file_name + '.gif'
    anim.save(gif_file, writer='imagemagick')
    plt.close(fig)

    # Optionally display gif in notebook
    if show:
        display(Image(filename=gif_file))

In [ ]:
def segment_video(frames, segments=8):
    """
    Segments video into equally sized segments and average all frames in each segment.
    
    Parameters:
        frames1 (np.ndarray): 3D array with shape (N, H, W).
        segments (int): number of segments to split frames into

    Returns
        Nothing, but plots the 8 averaged segments.
    """
    
    N_frames, height, width = frames.shape
    segment_size = N_frames // segments

    cols = 2
    rows = int(np.ceil(segments / cols)) # Calculate how many rows of subplots

    avg_frames = np.empty((segments, frames.shape[1], frames.shape[2]))
    for i in range(segments):
        start_frame = i * segment_size
        end_frame = (i + 1) * segment_size
        avg_frame = np.mean(frames[start_frame:end_frame, :, :], axis=0)
        avg_frames[i] = avg_frame

    fig, axs = plt.subplots(rows, cols, figsize=(cols*4, rows*4), constrained_layout=True)
    axs = axs.flatten()
    vmin = avg_frames.min()
    vmax = avg_frames.max()
    for i in range(avg_frames.shape[0]):
        start_frame = i * segment_size
        end_frame = (i + 1) * segment_size
        im = axs[i].imshow(avg_frames[i], cmap='viridis', vmin=vmin, vmax=vmax)
        if i == 0:
            cbar = fig.colorbar(im, ax=axs[i], location='left')
        axs[i].set_title(f'Frame {start_frame+1} - Frame {end_frame+1}')
        axs[i].axis('off')

In [ ]:
make_gif(aligned_avg)

In [ ]:
tiled = make_tiles(aligned_avg, tile_size = 25)

In [ ]:
make_gif(tiled)

In [ ]:
# ∆F/F where F0 calculated pixel by pixel
F0 = np.percentile(aligned_avg, 10, axis=0)
F1 = aligned_avg
deltaF_p = (F1-F0)/F0
make_gif(deltaF_p)

In [ ]:
# ∆F/F where F0 calculated frame by frame
F0 = np.percentile(aligned_avg, 10, axis=(1,2))
F1 = aligned_avg
deltaF_f = (F1 - F0[:, None, None]) / F0[:, None, None]
make_gif(deltaF_f)

In [ ]:
# ∆F/F where F0 calculated pixel by pixel on tiled image
# NB: This is what I'm using in subsequent ∆F/F timeseries plots/gifs
F0 = np.percentile(tiled, 10, axis=0)
F1 = tiled
deltaF_tiled_p = (F1-F0)/F0
make_gif(deltaF_tiled_p)

In [ ]:
# ∆F/F where F0 calculated frame by frame on tiled image
F0 = np.percentile(tiled, 10, axis=(1,2))
F1 = tiled
deltaF_tiled_f = (F1 - F0[:, None, None]) / F0[:, None, None]
make_gif(deltaF_tiled_f)

In [ ]:
# Define tiles with significant dynamics as those with std dev > 4 * median std dev
sd = np.std(tiled, axis=0)  # sd for each tile across time
avg_sd = np.median(sd)
threshold = 4 * avg_sd
significant_tiles = sd > threshold
all_tiles = sd > -1000

In [ ]:
show(significant_tiles)

In [ ]:
overlay_rois(aligned_avg, significant_tiles, tile_size=25)

In [ ]:
plot_rois(deltaF_tiled_p, significant_tiles, tile_size=25)

In [ ]:
# Overlaying timeseries where F0 for ∆F is calculated pixel-by-pixel
overlay_plotted_rois(aligned_avg, deltaF_tiled_p, significant_tiles, ymax=2)

In [ ]:
# Overlaying timeseries where F0 for ∆F is calculated frame-by-frame
overlay_plotted_rois(aligned_avg, deltaF_tiled_f, significant_tiles, ymax=5)

In [ ]:
# Pixel-by-pixel F0, all tiles
overlay_plotted_rois(aligned_avg, deltaF_tiled_p, all_tiles, ymax=3)

In [ ]:
# Frame-by-frame F0, all tiles
overlay_plotted_rois(aligned_avg, deltaF_tiled_f, all_tiles, ymax=5)

In [ ]:
intensity_time_map(aligned_avg)

In [ ]:
intensity_time_map(aligned_avg, tiles_to_plot=significant_tiles, tile_size=25)

In [ ]:
make_overlaid_timeseries_gif(aligned_avg, deltaF_tiled_p, significant_tiles, file_name='overlaid_timeseries_sig_F0p')

In [ ]:
make_overlaid_timeseries_gif(aligned_avg, deltaF_tiled_p, all_tiles, file_name='overlaid_timeseries_all_F0p')

In [ ]:
make_overlaid_timeseries_gif(aligned_avg, deltaF_tiled_f, all_tiles, ymax=5, file_name='overlaid_timeseries_all_F0f')

In [ ]:
segment_video(aligned, 8)